In [ ]:
import search as search 

import time
import matplotlib.pyplot as plt
import numpy as np


# Creating an AI Agent to Escape a Maze and Collect all the Treasure!

In this tutorial, we're going to use A* search to allow an agent to efficiently find a path through to escape a maze while collecting a set of treasure chests. The start position of the agent is shown in red, the exit is shown in green, and the treasure is shown in yellow. The agent can only exit after it has collected all treasure. The agent collects treasure by walking over it.

![alt text](treasure_maze.png)

We will build off the Week 9 tutorial, expanding our previous Maze_Solver into a Maze_Treasure_Solver! 

First, let's go to search.py and look through:

* astar_graph_search()


Similar to last week, we have a maze that is represented as a string. The maze we will solve has a single start 'S' and single exit 'G'. Walls are indicated with '#', and treasure is indicated with 'T'.

In [ ]:
maze1 = '''#####################
#       #       #  T#
### # # ### ### # ###
#T  # #     # # #   #
# ### ####### # ### #
#     #       #     #
### ### # ###########
#           #T#     #
# ### # ### # ### # #
# #   # S   #   # # #
# # ####### ### # # #
#           # # # # #
# ####### ### # # # #
#       #   #       #
## #### ######### ###
#   #             # #
# # # ### # ##### # #
#T    #     #   #   #
#################G###'''

## Using classes to store information about our world state

Our string representation above is intuitive to type out to draw a maze, but it's not very useful for an algorithm to use. Building off the Week 9 tutorial, this Maze class now also includes 2 new fields:
* self.treasure_loc -- stores the (x,y) coordinates of all treasure chests
* self.treasure_state -- stores the open (True)/unopened (False) state of all treasure chests

Read through the class to understand these changes.

In [ ]:
class Maze:
    '''
    Given a string of a maze, with # for walls, S for a starting position, T for treasure, and G for the goal,
    this class will extract the (x,y) location of walls, (x,y) location of the start, (x,y) location of all 
    treasure chests and (x,y) location of the goal. All treasure chests are initially unopened.
    You can also use this class to print the maze.
    '''
    def __init__(self, maze):

        self.maze = maze
        
        self.walls = []
        self.find_walls(maze)

        self.goal = None
        self.find_goal(maze)

        self.start = None
        self.find_start(maze)

        self.treasure_loc = []
        self.treasure_state = [] #this will store whether the treasure has been collected (True) or not (False)
        self.find_treasure(maze)
        
    def find_walls(self, maze):
        #store the (x,y) coordinates of all the walls in the maze
        rows = maze.split('\n')
        for y, line in enumerate(rows):
            self.walls += [(x, y) for x, letter in enumerate(line) if letter == '#']

    def find_goal(self, maze):
        #search for a 'G' and return position
        rows = maze.split('\n')
        for y, line in enumerate(rows):
            for x, char in enumerate(line):
                if char == 'G':
                    self.goal = (x, y)
                    return

        if self.goal == None:
            print('Error, no goal found.')

    def find_start(self, maze):
        #search for a 'S' and return position
        rows = maze.split('\n')
        for y, line in enumerate(rows):
            for x, char in enumerate(line):
                if char == 'S':
                    self.start = (x, y)

        if self.start == None:
            print('Error, no start found.')

    def find_treasure(self, maze):
        #search for a 'T' and store position 
        rows = maze.split('\n')
        for y, line in enumerate(rows):
            for x, char in enumerate(line):
                if char == 'T':
                    self.treasure_loc += [(x, y)]
                    self.treasure_state += [False]
                                        
    
    def print(self, m = None):
        #can print the string of any maze passed in, otherwise prints the stored maze
        if m == None:
            m = self.maze
        for line in m.split('\n'):
            print(''.join(line))

    def visualise(self, maze = None):
        #can print the string of any maze passed in, otherwise prints the stored maze
        if maze == None:
            maze = self.maze
            
        # Convert the maze string into a list of lists
        maze_rows = maze.split('\n')
        height = len(maze_rows)
        width = len(maze_rows[0])

        # Create a 2D numpy array to store the maze representation
        maze_array = np.ones((height, width, 3))
        
        # Fill the numpy array: 1 for walls ('#')
        for y, row in enumerate(maze_rows):
            for x, char in enumerate(row):
                if char == '#':
                    maze_array[y, x, :] = 0  # Wall
                if char == 'T':
                    maze_array[y, x] = [255,255,0]
                if char == 'S': 
                    maze_array[y,x] = [255, 0, 0]
                if char == 'G':
                    maze_array[y,x] = [0, 255, 0]
                if char == 'o':
                    maze_array[y, x, :] = 0.5  # Optional: Distinguish path as gray
        
        # Plot the maze using matplotlib
        plt.figure(figsize=(2, 2))
        plt.imshow(maze_array, cmap='gray')
        plt.axis('off')  # Hide the axis
        plt.title("Maze Visualization")
        plt.show()


Below, we can use our Maze class with our maze1 string to create a Maze instance.

In [ ]:
maze = Maze(maze1)
maze.visualise()

## Updating our Maze_Solver to a Maze_Treasure_Solver!

Below you can see the Maze_Solver class, which is a subclass of the search.Problem class. We created this class in the Week 9 tutorial.

**Consider: Which of the following class variables and methods will need to change for our new task? How should they change and why?** 
* self.goal?
* self.initial?
* actions()?
* result()?


In [ ]:
class Maze_Treasure_Solver(search.Problem):
    '''
    Your implementation should be fully compatible with the search functions of 
    the provided module 'search.py'. 
    '''
    def __init__(self,
                 maze,
                 ): 

        self.maze = maze
        self.goal = maze.goal 
        self.initial = maze.start 
       
        self.initial = tuple(self.initial)# use tuple to make the state hashable
        self.goal = tuple(self.goal) # use tuple to make the state hashable

        self.min_x = min([x for x,y in self.maze.walls])
        self.max_x = max([x for x,y in self.maze.walls])
        self.min_y = min([y for x,y in self.maze.walls])
        self.max_y = max([y for x,y in self.maze.walls])
        
    ## - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

    def actions(self, state):
        """
        Return the list of actions that can be executed in the given state.
        """
        possible_actions = []
        
        x, y = state
        if (x-1, y) not in self.maze.walls and x-1 >= self.min_x:
            possible_actions += ['left']

        if (x, y-1) not in self.maze.walls and y-1 >= self.min_y:
            possible_actions += ['up']

        if (x+1, y) not in self.maze.walls and x+1 <= self.max_x:
            possible_actions += ['right']

        if (x, y+1) not in self.maze.walls and y+1 <= self.max_y:
            possible_actions += ['down']
        
        return possible_actions


    def result(self, state, action):
        """
        Return the state that results from executing the given
        action in the given state. The action must be one of
        self.actions(state).

        The updated state should be returned as a tuple, i.e. return tuple(state), to make the state hashable
        """
        x, y = state
        if action == 'up':
            y -= 1
        if action == 'down':
            y += 1
        if action == 'left':
            x -= 1
        if action == 'right':
            x += 1
            
        return x, y
        
    def print_solution(self, goal_node):
        """
            Shows solution represented by a specific goal node.
            For example, goal node could be obtained by calling 
                goal_node = breadth_first_tree_search(problem)
        """
        # path is list of nodes from initial state (root of the tree)
        # to the goal_node
        path = goal_node.path()
        # print the solution
        print( f"Solution takes {len(path)-1} steps from the initial state to the goal state\n")
        print( "Below is the sequence of moves\n")
        moves = []
        for node in path:
            if node.action:
                moves += [f"{node.action}, "]
        print(moves)
            
        print( "Below is the path drawn on the maze\n")
        self.print_maze_solution(path)
        

    def print_maze_solution(self, path):
        s = self.maze.maze

        maze_rows = [list(row) for row in s.split('\n')]

        for node in path:
            x,y = node.state
            maze_rows[y][x] = 'o'
        
        s_path = '\n'.join([''.join(row) for row in maze_rows])
        self.maze.visualise(s_path)               


## Using Search to Solve the search.Problem class

Below, you can call breadth_first_graph_search and astar_graph_search on your Maze_Solver! 

1. First, you will need to make sure you have updated all relevant methods in the Maze_Treasure_Solver class for our new task.

2. Once you have done this, consider the BFS solution in terms of the number of steps they used and the time taken to find a solution.

3. Next, try to change search_type to 'astar'. It will throw an error because you have not yet created a heuristic function! Go back to the Maze_Treasure_Solver class and add a h() method.

4. With your new heuristic implemented, try to use astar search. How does the number of steps and efficiency compare to BFS?

In [ ]:
search_type = 'bfs'

solver = Maze_Treasure_Solver(maze)

t0 = time.time()
if search_type == 'bfs':
    # Solve with Breadth First Search
    sol_ts = search.breadth_first_graph_search(solver)
else:
    # Solve with A*
    sol_ts = search.astar_graph_search(solver)
t1 = time.time()

print (f"Solver took {1000*(t1-t0):.2f} milli-seconds to find a solution.")
solver.print_solution(sol_ts)

## Concluding Remarks

**Consider: Were the heuristics you tried admissable and consistent?**

**Consider: What could be a useful heuristic for Project 2?**